# MV plausibility

This notebook checks the MV grids of the MV variants of one LV version against external references:

1. published values and ranges of MV grid metrics (`pylovo.mv_grid.metrics.LITERATURE_RANGES`) as traffic lights;
2. the circuit length per km² and the cable share of the region parts;
3. the eGon-data MV grid districts (area, substation match);
4. the Ding0 2023 grids of the same districts (optional, `ding0_dir`);
5. the demand anchor (coincident MV load per inhabitant) and the MV customers;
6. the capacitive earth-fault current per district;
7. the pandapower example net `mv_oberrhein` with the same metric code.

All topology metrics come from the generic metric code of `pylovo.mv_grid.metrics`: the same functions run on the
MV variants, on the Ding0 grids and on pandapower nets. Rings and half-rings are detected from the line tables
(feeders = components without the busbar and the normally-open lines), so a half-ring here includes its
normally-open line and is slightly longer than `half_a_km`/`half_b_km` of `pylovo.mv_ring`.

**Prerequisites**

- pylovo with the `notebooks` extra (`uv sync --extra notebooks`) and this notebook running on the kernel of the
  project environment (`.venv`), see `notebook_tutorials/README.md`.
- A `.env` in the repository root that points to the database with the MV results, and MV variants of the LV
  version (`pylovo-mv list --version <version>`; `0_mv_generation.ipynb` shows how to generate them).
- The notebook only reads from the database: its client runs read-only transactions.
- For section 4: the Ding0 2023 MV grids (Zenodo record 10405129, licence CC BY-SA 4.0) as
  `<ding0_dir>/<bus_id>/topology/*.csv`, where `bus_id` is the eGon-data MV grid district.
- For section 3: the eGon-data tables `egon_hvmv_substation` and `egon_mv_grid_district` (`pylovo` schema after
  `pylovo-import egon`, or the opendata schema).

In [ ]:
# Parameters
version_id = None      # LV version; None: VERSION_ID of config/config_generation.yaml
variants = None        # MV variants, e.g. ["d0", "d1"]; None: all MV variants of the version
regions = None         # region parts to keep (1 = largest), e.g. [1]; None: all
plz = None             # keep only the districts with nodes in these PLZ; None: all
complete_only = True   # literature comparison on complete districts only (falls back to all when none is complete)
ding0_dir = None       # folder of the Ding0 2023 grids (<bus_id>/topology/*.csv); None: skip section 4
egon_min_share = 0.9   # eGon districts with at least this share of their area in the region are compared
dbname = None          # expected database name of the .env; None: no check
repo_root = None       # repository root; None: two levels above this notebook

## Setup

The first cell changes to the repository root, so that pylovo reads the `.env` and `config/` of this checkout
(and writes its log to `log/`), and checks the database name when `dbname` is set. `mv_notebook_utils.py` next to
this notebook holds the shared helpers of the MV notebooks.

In [ ]:
import os
import sys
from pathlib import Path

if repo_root is None:  # the notebook runs in notebook_tutorials/mv_grids of the repository
    here = Path.cwd().resolve()
    repo_root = here if (here / "pyproject.toml").is_file() else here.parents[1]
os.chdir(repo_root)
sys.path.insert(0, str(Path(repo_root) / "notebook_tutorials" / "mv_grids"))

from pylovo import config_loader

if dbname is not None:
    assert config_loader.DBNAME == dbname, f"the .env points to {config_loader.DBNAME!r}, not {dbname!r}"

import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

import mv_notebook_utils as mvu
from pylovo.mv_grid import metrics
from pylovo.plotting import mv_grid as mvplot
from pylovo.plotting.utils import get_color_map

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 400)
pd.set_option("display.max_colwidth", 200)

dbc = mvu.read_only_client()
version_id, variants, listing = mvu.resolve(dbc, version_id, variants)
print(f"Database {config_loader.DBNAME}, LV version {version_id}, MV variants: {', '.join(variants) or 'none'}")

In [ ]:
assert variants, "no MV variants to analyse"
plz_union = sorted({int(p) for p in listing.loc[listing["mv_variant_id"].isin(variants), "plz_list"].explode()})
parts = mvu.region_parts(dbc, plz_union)
data = {v: mvu.load_variant(dbc, version_id, v, parts=parts, regions=regions, plz=plz) for v in variants}
colors = get_color_map(variants)
districts = pd.concat([mvu.district_table(v) for v in data.values()], ignore_index=True)
rings = pd.concat([mvu.ring_table(v) for v in data.values()], ignore_index=True)
print(f"{len(parts)} region parts; districts per variant: "
      + ", ".join(f"{k} {len(v.grids)}" for k, v in data.items()))

from pylovo.mv_grid.parameters import LvVersionParameters

lv = LvVersionParameters.from_generation_parameters(version_id, dbc.lv_snapshot(version_id))
dbc.conn.rollback()
generic = {name: mvu.generic_rings_and_feeders(v) for name, v in data.items()}
g_rings = pd.concat([r for r, _ in generic.values()], ignore_index=True)
g_feeders = pd.concat([f for _, f in generic.values()], ignore_index=True)
print(f"Power factor of the loads (LV version): {lv.power_factor}")

## 1. Literature ranges

One row per variant and area class. Districts count by the majority class of their supply points, rings and
feeders (half-rings) by their own class, stations by their node class. Ring and feeder values are means,
`km_per_supply_point`, `cable_share`, `km_per_km2` (district cells) and `customer_share` ratios of sums,
per-substation values district means; `substation_peak_mva` divides the district peak by the power factor of the
loads. **Basis**: the complete districts (substation in the planning region, cell not cut by the region border);
a small region has none, then all districts count.

Traffic lights: green inside the range, amber within 30 % of the violated bound, red beyond. A reference that
gives only a typical value is compared as a point, so it is rarely green.

In [ ]:
lit = pd.concat([mvu.literature_table(v, districts, g_rings, g_feeders, lv.power_factor, complete_only)
                 for v in data.values()])
display(mvu.pretty(lit.drop(columns=["area_class"]).T, rows=True))
comparison = metrics.compare_to_literature(lit)
counts = (comparison[comparison["status"] != "n/a"].groupby(["variant", "class", "status"]).size()
          .unstack(fill_value=0).reindex(columns=["green", "amber", "red"], fill_value=0))
display(counts)
display(mvu.traffic_light_table(comparison))

In [ ]:
for name in variants:
    part = comparison[(comparison["variant"] == name) & (comparison["status"] != "n/a")]
    if len(part):
        fig = mvplot.plot_plausibility(part, title=f"{name}: plausibility against literature ranges "
                                       "(rural and urban rows side by side)")

**Distributions.** The share of the rings and half-rings of each variant that lie inside each range (same basis
as above).

In [ ]:
def basis_ids(name):
    d = districts[districts["variant"] == name]
    complete = d["complete"].fillna(False).astype(bool)
    if complete_only and complete.any():
        d = d[complete]
    return set(d["mv_grid_id"])


frames = []
for name in variants:
    ids = basis_ids(name)
    r = g_rings[(g_rings["variant"] == name) & g_rings["mv_grid_id"].isin(ids) & g_rings["kind"].isin(["ring", "mesh"])]
    f = g_feeders[(g_feeders["variant"] == name) & g_feeders["mv_grid_id"].isin(ids)
                  & g_feeders["kind"].isin(["ring", "mesh"]) & (g_feeders["feeder_supply_points"] > 0)]
    for frame, columns in ((r, ["ring_members", "ring_km"]), (f, ["feeder_supply_points", "feeder_km"])):
        if len(frame):
            table = frame[["area_class", *columns]].reset_index(drop=True)
            frames.append(metrics.compare_to_literature(table).assign(variant=name))
shares = pd.concat(frames, ignore_index=True)
shares = shares[shares["status"] != "n/a"]
shares = (shares.groupby(["metric", "area_class", "label", "variant"])["status"].value_counts(normalize=True)
          .unstack(fill_value=0).reindex(columns=["green", "amber", "red"], fill_value=0))
display(shares.round(2))

## 2. Region parts: circuit km per km² and cable share

The lines are clipped to the region parts (`metrics.region_metrics`; region 0 = all selected parts), as in the
structural data that DSOs publish for their whole grid. A region part is compared with the ranges of the class of
the majority of its supply points.

In [ ]:
clipped = mvu.clipped_table(list(data.values()))


def majority_class(d):
    urban = d.loc[d["area_class"] == "urban", "n_supply_points"].sum()
    rural = d.loc[d["area_class"] == "rural", "n_supply_points"].sum()
    return "urban" if urban > rural else "rural"


classes = {(name, 0): majority_class(d) for name, d in districts.groupby("variant")}
classes.update({key: majority_class(d) for key, d in districts.groupby(["variant", "region"])})
clipped["area_class"] = [classes.get(key) for key in clipped.index]
display(mvu.pretty(clipped))
region_comparison = metrics.compare_to_literature(clipped[["area_class", "km_per_km2", "cable_share",
                                                           "km_per_supply_point"]])
region_comparison["class"] = [f"region {r} ({classes.get((v, r))})" for v, r in zip(region_comparison["variant"],
                                                                                    region_comparison["region"])]
display(mvu.traffic_light_table(region_comparison))

## 3. eGon-data MV grid districts

eGon-data derives its MV grid districts from Voronoi cells around the HV/MV substations (Hülk et al. 2017). For
every eGon district that intersects the selected region parts: its share inside the region, the districts of the
variant whose substation matches the eGon substation (`egon_bus_id` within `egon_match_m`, or the eGon source of
the variant), their cell area and the intersection over union (IoU) of their cells with the eGon district inside
the region. `substation_distance_m`: eGon substation to the nearest substation of the variant.

In [ ]:
import geopandas as gpd
from matplotlib.lines import Line2D

egon = {name: mvu.egon_comparison(dbc, v) for name, v in data.items()}
for name, e in egon.items():
    display(Markdown(f"**{name}**"))
    table = e.drop(columns=e.geometry.name).set_index("bus_id")
    table["mv_grid_ids"] = table["mv_grid_ids"].map(lambda ids: ", ".join(map(str, ids)))
    display(mvu.pretty(table))
    inside = e[e["share_in_region"] >= egon_min_share]
    print(f"{name}: {len(e)} eGon districts touch the region, {len(inside)} with at least {egon_min_share:.0%} "
          f"inside; {int(inside['mv_grid_ids'].map(bool).sum())} of these matched by a substation of the variant, "
          f"mean IoU {inside['iou'].mean():.2f}")

In [ ]:
for name, v in data.items():
    e = egon[name]
    fig, ax = plt.subplots(figsize=(12, 12))
    region = mvu.selected_region(v)
    gpd.GeoSeries([region]).plot(ax=ax, color="#eeeeee", zorder=0)
    e.boundary.plot(ax=ax, color="#d62728", linewidth=1.3, zorder=2)
    v.grids.boundary.plot(ax=ax, color="#1f77b4", linewidth=1.0, linestyle="--", zorder=3)
    subs = e.attrs["substations"]
    ax.scatter(subs["x"], subs["y"], marker="x", s=70, color="#d62728", zorder=5)
    used = v.substations[v.substations["used"]]
    ax.scatter(used.geometry.x, used.geometry.y, marker="^", s=90, color="#2c3e50", zorder=6)
    for bus, point in zip(e["bus_id"], e.geometry.representative_point()):
        ax.annotate(str(bus), (point.x, point.y), fontsize=8, color="#d62728", ha="center")
    ax.legend(handles=[Line2D([], [], color="#d62728", label="eGon-data MV grid district"),
                       Line2D([], [], color="#1f77b4", linestyle="--", label=f"district cell of {name}"),
                       Line2D([], [], marker="x", color="#d62728", linestyle="none", label="eGon-data substation"),
                       Line2D([], [], marker="^", color="#2c3e50", linestyle="none", label=f"substation of {name}")],
              loc="upper right", fontsize=9)
    x0, y0, x1, y1 = region.buffer(3000).bounds
    ax.set_xlim(x0, x1)
    ax.set_ylim(y0, y1)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(f"{name}: district cells against eGon-data MV grid districts (grey: planning region)",
                 fontweight="bold")

## 4. Ding0 2023 grids of the same districts

Ding0 generates MV grids per eGon-data district (open rings planned with a savings and local-search heuristic on
straight lines with a detour factor). The Ding0 2023 grids are read with `metrics.read_ding0_grid` (MV buses,
stations = MV side of the MV/LV transformers, busbar = MV side of the HV/MV transformers, `switches.csv` = open
points) and compared with the district of the variant whose substation matches the eGon substation. The districts
differ in area unless the variant uses the eGon districts (`MV_SUPPLY_AREAS=egon`), see the IoU of section 3;
per-km² and per-station values compare better than totals. The Ding0 grids are under CC BY-SA 4.0.

In [ ]:
from pathlib import Path

ding0, grids = pd.DataFrame(), {}
if ding0_dir is None:
    print("ding0_dir is not set: the Ding0 comparison is skipped")
else:
    buses = sorted({b for e in egon.values() for b in e.loc[e["share_in_region"] >= egon_min_share, "bus_id"]
                    if (Path(ding0_dir) / str(b)).is_dir()})
    rows = []
    for bus in buses:
        dn, dl = metrics.read_ding0_grid(Path(ding0_dir) / str(bus))
        district = mvu.read_ding0_district(Path(ding0_dir) / str(bus))
        grids[bus] = (dn, dl, district)
        rows.append({"bus_id": bus, "source": "Ding0 2023", **mvu.grid_metrics(dn, dl),
                     "area_km2": district["area_km2"], "population": district["population"]})
    for name, v in data.items():
        e = egon[name].set_index("bus_id")
        cells = v.grids.set_index("mv_grid_id")
        for bus in buses:
            ids = e.at[bus, "mv_grid_ids"] if bus in e.index else []
            if ids:
                rows.append({"bus_id": bus, "source": name, **mvu.grid_metrics(*mvu.generic_tables(v, ids)),
                             "vn_kv": float(cells.loc[ids, "vn_kv"].max()),
                             "area_km2": float(cells.loc[ids, "area_km2"].sum()), "iou": e.at[bus, "iou"]})
    ding0 = pd.DataFrame(rows)
    if ding0.empty:
        print(f"no Ding0 grid in {ding0_dir} for the eGon districts with {egon_min_share:.0%} inside the region")
    else:
        ding0["km_per_km2"] = ding0["circuit_km"] / ding0["area_km2"]
        display(mvu.pretty(ding0.set_index(["bus_id", "source"]).T, rows=True))

**Ratios to Ding0** per district and over all compared districts (sums). The open_eGo final report found Ding0
district lengths at a median of about 0.44 of the DSO lengths (`LITERATURE_RANGES`, `length_ratio_to_real`); a
ratio well above 1 is expected for street-routed grids.

In [ ]:
if not ding0.empty:
    reference = ding0[ding0["source"] == "Ding0 2023"].set_index("bus_id")
    ratio_rows = []
    for name in variants:
        ours = ding0[ding0["source"] == name].set_index("bus_id")
        common = ours.index.intersection(reference.index)
        if common.empty:
            continue
        for bus in common:
            ratio_rows.append({"variant": name, "bus_id": bus, **{
                m: ours.at[bus, m] / reference.at[bus, m] if reference.at[bus, m] else np.nan
                for m in ("n_stations", "circuit_km", "km_per_station", "km_per_km2", "ring_members_mean",
                          "feeder_km_mean", "cable_share", "area_km2")}, "iou": ours.at[bus, "iou"]})
        pooled = {m: ours.loc[common, m].sum() / reference.loc[common, m].sum()
                  for m in ("n_stations", "circuit_km", "area_km2")}
        ratio_rows.append({"variant": name, "bus_id": "all", **pooled,
                           "km_per_station": pooled["circuit_km"] / pooled["n_stations"],
                           "km_per_km2": pooled["circuit_km"] / pooled["area_km2"]})
    ratios = pd.DataFrame(ratio_rows).set_index(["variant", "bus_id"])
    display(ratios.round(2))
    print("Voltage levels: " + ", ".join(f"{s} {b}: {kv:g} kV" for s, b, kv in
                                          zip(ding0["source"], ding0["bus_id"], ding0["vn_kv"])))

In [ ]:
if not ding0.empty:
    first = data[variants[0]]
    e = egon[variants[0]]
    for bus, (dn, dl, district) in grids.items():
        ids = e.loc[e["bus_id"] == bus, "mv_grid_ids"].iloc[0]
        fig, axes = plt.subplots(1, 2, figsize=(20, 10))
        if ids:
            mvplot.plot_mv_grid_map(first.layers, ids[0], color_by="ring", ax=axes[0], settlements=False,
                                    title=f"{variants[0]}: district {ids[0]} (eGon substation {bus})")
        mvu.plot_generic_grid(dn, dl, ax=axes[1], title=f"Ding0 2023: district {bus} ({district['area_km2']:.0f} km²)")
        bounds = [district["geometry"].bounds]
        if ids:
            bounds.append(first.grids.set_index("mv_grid_id").geometry[ids[0]].bounds)
        x0, y0 = min(b[0] for b in bounds), min(b[1] for b in bounds)
        x1, y1 = max(b[2] for b in bounds), max(b[3] for b in bounds)
        for ax in axes:
            for poly in getattr(district["geometry"], "geoms", [district["geometry"]]):
                ax.plot(*poly.exterior.xy, color="#d62728", linewidth=1.2, zorder=1)
            ax.set_xlim(x0 - 500, x1 + 500)
            ax.set_ylim(y0 - 500, y1 + 500)
        fig.tight_layout()

## 5. Demand anchor and MV customers

The coincident MV load per inhabitant (pylovo's coincidence law on all stations, MV customer nodes and
station-busbar customers of the PLZ; population of `municipal_register`). There is no published reference range;
the value shows how much load the LV stage hands to the MV stage. `customer_load_share` is the share of the fixed
MV customer loads in the coincident load. HV customers (above `hv_customer_kw`) are not modelled
(`excluded_mw`).

In [ ]:
anchors = pd.concat([mvu.demand_anchors(dbc, v) for v in data.values()], ignore_index=True)
anchors["customer_load_share"] = (anchors["customer_mw"] + anchors["busbar_customer_mw"]) / anchors["coincident_mw"]
display(mvu.pretty(anchors.set_index(["variant", "scope", "key"])))

## 6. Capacitive earth-fault current

`metrics.earth_fault_current_a` with the earth capacitance of the variant's cable and overhead line types (flag
only: districts with a large current would need several galvanically separated parts or earthing measures in
practice).

In [ ]:
earth = pd.concat([mvu.earth_fault_table(v, districts[districts["variant"] == name]) for name, v in data.items()],
                  ignore_index=True)
display(earth.groupby(["variant", "vn_kv"])["earth_fault_a"].describe().round(0))
fig = mvplot.plot_metric_distribution(earth, "earth_fault_a", by="variant", order=variants, figsize=(10, 5),
                                      title="Capacitive earth-fault current per district (A)")

## 7. Reference net: pandapower `mv_oberrhein`

The pandapower example MV net (bundled with pandapower, offline) through the same metric code: a sanity check of
the generic format and one more reference point for the topology metrics of the variants (all districts of a
variant together; per-substation values are means over the substations). The example net has two substations
with one large open ring each and three normally-open ties between them (the hand count of
`tests/test_mv_metrics.py`), so its ring sizes are no planning reference.

In [ ]:
import pandapower.networks as pn

net = pn.mv_oberrhein()
on, ol = metrics.net_to_tables(net)
reference_rows = {"mv_oberrhein": mvu.grid_metrics(on, ol)}
for name, v in data.items():
    reference_rows[name] = mvu.grid_metrics(*mvu.generic_tables(v))
display(mvu.pretty(pd.DataFrame(reference_rows), rows=True))